# Saliency Detection

**Saliency detection** predicts which regions of an image are **visually most important** — the parts that would draw a human observer's attention first. The output is typically a single-channel **saliency map** the size of the image, with values in `[0, 1]` indicating how salient each pixel is.

There are two related but distinct tasks:

- **Eye-fixation / attention prediction**: model where human gaze lands (sparse, blurry hotspots). Evaluated with metrics like AUC, NSS, CC.
- **Salient object detection (SOD)**: produce a clean binary-like mask of the *whole* dominant object(s). This is the variant most relevant to segmentation pipelines and is the focus here.

## Saliency vs. semantic segmentation

| | Salient object detection | Semantic segmentation |
|---|---|---|
| **Question** | *What stands out?* | *What class is each pixel?* |
| **Classes** | Class-agnostic (salient vs. not) | Fixed label set (cat, road, sky, ...) |
| **Output** | One saliency/foreground map | One-hot per-class map |
| **Use** | Foreground extraction, cropping, background removal, image retargeting | Scene understanding, per-class masks |

SOD is essentially a **binary, class-agnostic foreground segmentation** driven by visual prominence rather than a category label. It is widely used as a preprocessing step (e.g., automatic background removal, thumbnailing, weakly-supervised segmentation).

## Classical vs. deep approaches

**Classical (pre-deep-learning)** methods rely on hand-crafted cues:

- **Contrast-based**: a region is salient if it differs strongly from its surroundings in color, intensity, or texture (e.g., Itti-Koch, histogram/region contrast).
- **Center prior / background prior**: objects tend to be near the center; image borders are usually background.
- **Spectral residual**: saliency from the log-amplitude spectrum of the Fourier transform.

These are fast but brittle on cluttered scenes.

**Deep approaches** learn saliency end-to-end with encoder-decoder CNNs that fuse multi-level features (low-level edges + high-level semantics):

- **U2-Net** — a nested U-structure ("U-squared") of Residual U-blocks (RSU) that captures rich multi-scale context at high resolution; the engine behind many background-removal tools.
- **BASNet** — a predict-then-**refine** architecture with a boundary-aware **hybrid loss** (BCE + SSIM + IoU) that sharpens edges.
- **PoolNet, EGNet, F3Net** — other strong SOD backbones emphasizing global pooling, edge guidance, or feature-fusion balancing.

## Metrics for salient object detection

Given a predicted saliency map `S` (values in `[0,1]`) and a binary ground-truth mask `G`:

- **MAE (Mean Absolute Error)** — average per-pixel difference; **lower is better**:
  $$\text{MAE} = \frac{1}{H\,W}\sum_{x,y} |S(x,y) - G(x,y)|$$

- **F-measure (F-beta)** — harmonic mean of precision and recall after thresholding, weighting precision more (`beta^2 = 0.3` by convention); **higher is better**:
  $$F_\beta = \frac{(1+\beta^2)\,\text{Precision}\cdot\text{Recall}}{\beta^2\,\text{Precision} + \text{Recall}}$$

- **S-measure (structure)** and **E-measure (enhanced-alignment)** — capture region/structure and pixel+global similarity, addressing weaknesses of pixel-only MAE.

```python
import torch

def mae(pred, gt):                       # pred, gt in [0,1], shape [H, W]
    return (pred - gt).abs().mean()

def f_measure(pred, gt, thr=0.5, beta2=0.3):
    p = (pred >= thr).float()
    tp = (p * gt).sum()
    precision = tp / (p.sum() + 1e-8)
    recall    = tp / (gt.sum() + 1e-8)
    return (1 + beta2) * precision * recall / (beta2 * precision + recall + 1e-8)
```

## Conceptual usage

SOD models are not in `torchvision`, but the published checkpoints (e.g., **U2-Net**) follow a simple pattern: feed a normalized RGB tensor, get a single-channel map, then threshold or use it directly as an alpha matte.

```python
# Pseudocode for a typical U2-Net-style salient object detector
import torch
import torch.nn.functional as F

model = load_u2net("u2net.pth").eval()          # external repo / checkpoint

x = preprocess(image).unsqueeze(0)              # [1, 3, 320, 320], normalized
with torch.no_grad():
    d0, *side_outputs = model(x)                # multiple side outputs; d0 = fused map

sal = torch.sigmoid(d0)[0, 0]                   # [H, W] saliency in [0,1]
sal = (sal - sal.min()) / (sal.max() - sal.min() + 1e-8)   # min-max normalize

mask = (sal > 0.5).float()                      # binary foreground
# Use `sal` as an alpha channel for background removal, or `mask` for a cutout.
```

Deep SOD networks output **several side maps** at different decoder depths and supervise all of them (deep supervision); at inference the fused map (`d0`) is the final prediction.

## References

- Qin et al., *U2-Net: Going Deeper with Nested U-Structure for Salient Object Detection*, 2020 — https://arxiv.org/abs/2005.09007
- Qin et al., *BASNet: Boundary-Aware Salient Object Detection*, 2019 — https://arxiv.org/abs/2101.04704
- Cheng et al., *Global Contrast based Salient Region Detection*, 2011 (classical) — https://mmcheng.net/salobj/
- Fan et al., *Structure-measure: A New Way to Evaluate Foreground Maps*, 2017 — https://arxiv.org/abs/1708.00786
- Survey: Borji et al., *Salient Object Detection: A Survey* — https://arxiv.org/abs/1411.5878